In [1]:
from gpt2_mlx import GPT2, DataLoader, ShardedDataLoader, GPTConfig
from train_gpt2_mlx import GPTTrainer
from prepare_data import DATASET_REGISTRY
import mlx.core as mx
import mlx.nn as nn
import os

In [3]:
MAX_ITRS = 5

config = GPTConfig(
    context_len=1024,
    batch_size=8,
    emb_dim=768,
    n_heads=12,
    n_transformer_blocks=12,
    dropout=0.05,
    vocab_size=50304,       # 50257 padded to nearest multiple of 128
    use_flash_attn=True,
    warmup_iters=10,
    learning_rate=6e-4,
    max_train_iters=MAX_ITRS,
    min_lr=6e-5,
    weight_decay=0.1,
    total_batch_size=524288,  # 2^19
)
assert config.total_batch_size % (config.batch_size * config.context_len) == 0
config.grad_acum_steps

64

In [4]:
model = GPT2(config)
# Force initialization — MLX is lazy, params aren't materialized until evaluated
mx.eval(model.parameters())
print(model)

GPT2(
  (wte): Embedding(50304, 768)
  (wpe): Embedding(1024, 768)
  (h.0): Block(
    (ln_1): LayerNorm(768, eps=1e-05, affine=True)
    (attn): CausalSelfAttention(
      (c_attn): Linear(input_dims=768, output_dims=2304, bias=True)
      (c_proj): Linear(input_dims=768, output_dims=768, bias=True)
    )
    (ln_2): LayerNorm(768, eps=1e-05, affine=True)
    (mlp): MLP(
      (c_fc): Linear(input_dims=768, output_dims=3072, bias=True)
      (gelu): GELU()
      (c_proj): Linear(input_dims=3072, output_dims=768, bias=True)
    )
  )
  (h.1): Block(
    (ln_1): LayerNorm(768, eps=1e-05, affine=True)
    (attn): CausalSelfAttention(
      (c_attn): Linear(input_dims=768, output_dims=2304, bias=True)
      (c_proj): Linear(input_dims=768, output_dims=768, bias=True)
    )
    (ln_2): LayerNorm(768, eps=1e-05, affine=True)
    (mlp): MLP(
      (c_fc): Linear(input_dims=768, output_dims=3072, bias=True)
      (gelu): GELU()
      (c_proj): Linear(input_dims=3072, output_dims=768, bias=Tru

In [5]:
# Cast model weights to bfloat16.
# Unlike PyTorch there's no autocast context — you cast weights once before training.
# set_dtype is available in MLX >= 0.18.
#
# Note: in this MLX version the predicate receives the array's dtype (not the module),
# so we can only filter by dtype — not by module type (e.g. can't skip LayerNorm).
# predicate=lambda dtype: dtype == mx.float32 ensures we only cast float32 arrays
# and leave int/other dtypes untouched.
model.set_dtype(mx.bfloat16, predicate=lambda dtype: dtype == mx.float32)
mx.eval(model.parameters())
print("Model weights cast to bfloat16")

# --- Reference: manual equivalent (pre MLX 0.18) ---
# def cast_to_bf16(params):
#     if isinstance(params, mx.array):
#         return params.astype(mx.bfloat16) if params.dtype == mx.float32 else params
#     elif isinstance(params, dict):
#         return {k: cast_to_bf16(v) for k, v in params.items()}
#     elif isinstance(params, list):
#         return [cast_to_bf16(v) for v in params]
#     return params
# model.update(cast_to_bf16(model.parameters()))
# mx.eval(model.parameters())

Model weights cast to bfloat16


In [6]:
# Count parameters (weight-tied lm_head shares wte.weight so counted once)
total_params = sum(v.size for _, v in nn.utils.tree_flatten(model.parameters()))
print(f"Total parameters: {total_params:,}")

Total parameters: 124,475,904


In [7]:
# Sanity check — single forward pass
x = mx.zeros((2, 16), dtype=mx.int32)
logits, loss = model(x, targets=x)
mx.eval(logits, loss)
print(f"logits shape: {logits.shape}")  # (2, 16, 50257)
print(f"loss: {loss.item():.4f}")       # ~10.8 (log(50257)) for random weights

logits shape: (2, 16, 50304)
loss: 9.2774


In [7]:
PROJECT_ROOT = "/Users/ashritkuma.samudrala/lnex/ex_llm_rag"
BOOKS_DIR    = f"{PROJECT_ROOT}/main/resources/books"

with open(f"{BOOKS_DIR}/tinyshakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(f"Total chars: {len(text)}")
print(text[:50])

Total chars: 1115394
First Citizen:
Before we proceed any further, hear


In [ ]:
loader_args = dict(batch_size=config.batch_size, context_len=config.context_len)
data_loader = DataLoader(text, **loader_args, split="train")
val_loader  = DataLoader(text, **loader_args, split="val")
test_loader = DataLoader(text, **loader_args, split="test")

In [8]:
# ── ShardedDataLoader ────────────────────────────────────────────
# Use this instead of the DataLoader cell above when training on pre-tokenized
# shards from prepare_data.py (FineWeb 55% + Books 25% + TinyStories 20%).
#
# Run prepare_data.py first:
#   python prepare_data.py --max_tokens 10_000_000    # quick test
#   python prepare_data.py                             # full datasets
#
# Then uncomment below and comment out the DataLoader cell above.

PROJECT_ROOT = os.path.dirname(os.path.abspath("__file__"))
data_dir = os.path.join(PROJECT_ROOT, "data")
dataset_weights = {name: cfg["weight"] for name, cfg in DATASET_REGISTRY.items()}
loader_args = dict(data_dir=data_dir, dataset_weights=dataset_weights,
                   batch_size=config.batch_size, context_len=config.context_len)
data_loader = ShardedDataLoader(**loader_args, split="train")
val_loader  = ShardedDataLoader(**loader_args, split="val")
test_loader = ShardedDataLoader(**loader_args, split="test")

ShardedDataLoader [train]: 'fineweb' — 1 shard(s), 90,000,000 tokens
ShardedDataLoader [train]: 'books' — 1 shard(s), 90,000,000 tokens
ShardedDataLoader [train]: 'tinystories' — 1 shard(s), 90,000,000 tokens
ShardedDataLoader [val]: 'fineweb' — 1 shard(s), 5,000,000 tokens
ShardedDataLoader [val]: 'books' — 1 shard(s), 5,000,000 tokens
ShardedDataLoader [val]: 'tinystories' — 1 shard(s), 5,000,000 tokens
ShardedDataLoader [test]: 'fineweb' — 1 shard(s), 5,000,000 tokens
ShardedDataLoader [test]: 'books' — 1 shard(s), 5,000,000 tokens
ShardedDataLoader [test]: 'tinystories' — 1 shard(s), 5,000,000 tokens


In [9]:
trainer = GPTTrainer(model, config, data_loader,
                     val_loader=val_loader, test_loader=test_loader)

Total trainable parameters: 124,475,904
Skipping mx.compile (grad_acum_steps = 64 > 1, using accumulation loop)


In [10]:
total_tox = data_loader.total_tokens()["total"]
tokes_per_itr = config.batch_size * config.context_len * config.grad_acum_steps
print(f"Will train on mps:\
     \nRuns {config.max_train_iters} training steps and each step runs for {config.grad_acum_steps} iterations processing {tokes_per_itr} tokens")

print("Token distribution  "+str(data_loader.total_tokens()))
print(f"Total training tokens {data_loader.total_tokens()["total"]}, requires min {data_loader.total_tokens()["total"] // tokes_per_itr} steps or {data_loader.total_tokens()["total"] // (config.batch_size * config.context_len )} iterations (microsteps)")

Will train on mps:     
Runs 5 training steps and each step runs for 64 iterations processing 524288 tokens
Token distribution  {'fineweb': 90000000, 'books': 90000000, 'tinystories': 90000000, 'total': 270000000}
Total training tokens 270000000, requires min 514 steps or 32958 iterations (microsteps)


In [11]:
trainer.train_model(time_itr=True)

Step 0: val_loss = 10.9769
Step 0,  loss = 10.9742 | dt = 130083.45 ms | norm = 17.1250 | LR = 6.00e-05 | token_throughput = 4030.40 tokens/s
Step 1,  loss = 10.1501 | dt = 121586.54 ms | norm = 8.6875 | LR = 1.20e-04 | token_throughput = 4312.06 tokens/s
Step 2,  loss = 9.7463 | dt = 123140.91 ms | norm = 4.7188 | LR = 1.80e-04 | token_throughput = 4257.63 tokens/s
Step 3,  loss = 9.5115 | dt = 123854.30 ms | norm = 3.7188 | LR = 2.40e-04 | token_throughput = 4233.10 tokens/s
Step 4,  loss = 9.3650 | dt = 124220.58 ms | norm = 3.6875 | LR = 3.00e-04 | token_throughput = 4220.62 tokens/s

Test loss = 9.0889
Checkpoint saved: out/ckpt_train  (237.4 MB)
Checkpoint saved: out/ckpt_train_inference  (237.4 MB)


In [14]:
# Sample from the trained model
GPT2.sample(model, start_text="Hello I'm a language model,", max_len=50, n_samples=1, seed=42,
            temperature=0.8, top_k=40)

Hello I'm a language model,... of was, andLaunch,,,,,, 
. the the
.,, the the the ,,,,. the, the,.,,.,,,,


["Hello I'm a language model,... of was, andLaunch,,,,,, \n. the the\n.,, the the the ,,,,. the, the,.,,.,,,,"]

In [ ]:
# Checkpoint — save model weights + metadata
ROOT = "/Users/ashritkuma.samudrala/lnex/ex_llm_rag"
ckpt_path = f"{ROOT}/main/resources/models/gpt2_mlx/ckpt_train"
trainer.checkpoint_path = ckpt_path
trainer.save_checkpoint(step=config.max_train_iters - 1)
print(ckpt_path)

In [ ]:
import mlx.core as mx
print(mx.__version__) 

SyntaxError: invalid syntax (425310226.py, line 1)